In [1]:
import json
from tqdm import tqdm
# standard library imports
import random
import json
import os
# third party imports
from unsloth import FastLanguageModel, is_bfloat16_supported
from trl import SFTTrainer
from transformers import TrainingArguments, BitsAndBytesConfig
from huggingface_hub import login
from datasets import Dataset

import pandas as pd
import numpy as np
import torch

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


## Section 3a

In [ ]:
#### Configuration
max_seq_length = 4096

# None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
dtype = torch.bfloat16    ## For DGX Spark we need torch.bfloat16

load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

### Select the model
model_tag = "Mistral-Nemo-Base-2407-bnb-4bit"
model_name = f"unsloth/{model_tag}" 

### Delete for distribution
hg_key = "secret"

In [3]:
## This is necessary for DGX spark
## Also if I fine-tuning a 4-bit model, i need to load_in_4bit
quantization_config = BitsAndBytesConfig(load_in_4bit=load_in_4bit) 

## See the docstring of BitsAndBytesConfig
device_map={'':torch.cuda.current_device()}    ## Again for DGX-SPARK

In [4]:
## This will start the student model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    quantization_config=quantization_config, #New line applicable only on DGX Spark
    device_map=device_map,   # or "cuda". #New line applicable only on DGX Spark
    token = hg_key,
    
)


==((====))==  Unsloth 2026.1.4: Fast Mistral patching. Transformers: 4.57.1. vLLM: 0.1.dev1+g18d4e481d.d20260108.cu130.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 121.628 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0a0+b558c986e8.nv25.11. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [5]:
# 2nd piece of code
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context # 4x longer contexts auto supported!
    random_state = 7723,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

# Finishes attaching LoRA adapters. After this, model is still “the same base model,” 
# but now it has trainable LoRA components so fine‑tuning is practical on a single GPU.

Unsloth 2026.1.4 patched 40 layers with 40 QKV layers, 40 O layers and 40 MLP layers.


## Section 1: Data Preparation

In [6]:
## Fodler where our input data is
data_folder   = "data"
prompt_folder = "prompts"

In [7]:
## Load the raw data: JSON OUTPUT GENERATED BY THE 70B TEACHER MODEL
json_path = os.path.expanduser(f"{data_folder}/Llama3-70B-full-output.json")

with open(json_path, 'r') as f:
    llama_results = json.load(f)

llama_results[1]

{'location': 'Edinburgh',
 'latitude': '55.94498320836646',
 'longitude': '-3.185293348616568',
 'description': 'Upper level of duplex. Boho rustic-chic former warehouse Loft located in lively University quarter of City centre for exploring historic Old Town and at heart of Fringe, Jazz and International Festival venues. Great location for exploring, eating out and good pubs all year round (NB owner’s suite on lower level).',
 'nearby': {'specific_locations': ['Old Town', 'University quarter'],
  'general_locations': ['city centre', 'pubs'],
  'parent_locations': ['University quarter']},
 'last_scraped': '2024-02-18',
 'id': 24288,
 'host_id': 46498}

In [8]:
# Get the instructions prompt for the teacher model, through this path where we find the instruction_template.txt file.
instr_path = os.path.expanduser(f"{prompt_folder}/instruction_template.txt")

with open(instr_path, 'r') as f:
    instruction= f.read()

instruction = instruction.replace('<location>', 'Edinburgh, UK')
print(instruction)

You will be given a short description from an Airbnb listing in Edinburgh. It may describe either:
- the property itself, or
- the surrounding neighbourhood.

Your task is to identify **places or locations described as being near** the property — for example, anything mentioned as a short walk away, close by, or nearby.

You should categorise all such places into three groups:

1. `specific_locations` — locations that are **named and mappable**, such as landmarks, parks, venues, or neighbourhood features (e.g. "The Meadows", "Ocean Terminal").
2. `general_locations` — vague or generic references to places that **are not named or are too broad to geocode**, such as "train station", "shops", or "city centre".
3. `parent_locations` — the neighbourhood or area where the property is located (e.g. "Marchmont", "Leith"). Do not include these in the other two categories unless the location is explicitly described as a separate nearby place. 

Correct minor spelling mistakes for named places (e

### Clean the data

In [9]:
data = []

# Ensure llama_results from the Teacher model is iterable
if isinstance(llama_results, dict):
    llama_results = [llama_results]

for result in tqdm(llama_results):
    out = {
        "instruction": instruction,
        "input": result.get("description", "")
    }

    nearby = result.get("nearby", {})

    # 🧠 If nearby is a string, try to parse it
    if isinstance(nearby, str):
        try:
            nearby = json.loads(nearby)
        except json.JSONDecodeError:
            # If parsing fails, treat as empty
            nearby = {}

    # 1️⃣ Handle missing or empty nearby entries
    if not isinstance(nearby, dict) or all(len(v) == 0 for v in nearby.values()):
        out["response"] = json.dumps([
            {"specific_locations": [], "general_references": []}
        ])
        data.append(out)
        continue

    # 2️⃣ Extract lists safely
    specific = nearby.get("specific_locations", [])
    general = nearby.get("general_locations", [])
    parent  = nearby.get("parent_locations", [])

    # Combine general + parent → general_references
    general_refs = list(set(general + parent))  # optional deduplication

    # 3️⃣ Create structured response
    structured_response = [
        {
            "specific_locations": specific,
            "general_references": general_refs,
            "parent_references": parent
        }
    ]

    # 4️⃣ Serialize to JSON string
    out["response"] = json.dumps(structured_response)
    data.append(out)

# ✅ Example check
print(data[0])
print(len(data))

100%|██████████| 9201/9201 [00:00<00:00, 314821.72it/s]

{'instruction': 'You will be given a short description from an Airbnb listing in Edinburgh. It may describe either:\n- the property itself, or\n- the surrounding neighbourhood.\n\nYour task is to identify **places or locations described as being near** the property — for example, anything mentioned as a short walk away, close by, or nearby.\n\nYou should categorise all such places into three groups:\n\n1. `specific_locations` — locations that are **named and mappable**, such as landmarks, parks, venues, or neighbourhood features (e.g. "The Meadows", "Ocean Terminal").\n2. `general_locations` — vague or generic references to places that **are not named or are too broad to geocode**, such as "train station", "shops", or "city centre".\n3. `parent_locations` — the neighbourhood or area where the property is located (e.g. "Marchmont", "Leith"). Do not include these in the other two categories unless the location is explicitly described as a separate nearby place. \n\nCorrect minor spelling

### Section 2a: Prompt formating

In [10]:
# Set up the prompt template for model training

ft_prompt = """Below is an instruction that describes a task, paired with an input that provides a specfic example which the task should be applied to. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}
"""

EOS_TOKEN = tokenizer.eos_token # Must add EOS_TOKEN

# Chrysa: We now convert the prompt  into a instruction-input-ouput format :
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["response"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        # Must add EOS_TOKEN, otherwise your generation will go on forever!
        text = ft_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

ft_data = {"items":data}
#print(ft_data['items'][0]) # Let's see the first transformed row

### Section 2b: Data split

In [11]:
# Set up the test/train split
random.seed(7723)
trn_idxs = random.sample(range(len(data)), 6500) #2200
val_idxs = [x for x in range(len(data)) if x not in trn_idxs]
trn_data = [data[i] for i in trn_idxs]
val_data = [data[i] for i in val_idxs]

trn_dataset = Dataset.from_pandas(pd.DataFrame(trn_data), split="train")
print(trn_dataset.shape)

# I assume for validation dataset 
val_dataset = Dataset.from_pandas(pd.DataFrame(val_data), split="validation")
print(val_dataset.shape)

(6500, 3)
(2701, 3)


In [12]:
val_dataset

Dataset({
    features: ['instruction', 'input', 'response'],
    num_rows: 2701
})

In [13]:
# format data for input into the model
trn_dataset = trn_dataset.map(formatting_prompts_func, batched = True,)

Map:   0%|          | 0/6500 [00:00<?, ? examples/s]

### Section 3b: Train the Student model

In [14]:
## Training
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = trn_dataset,
    eval_dataset = None, # Chrysa: Can set up evaluation!
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        num_train_epochs = 3,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4, # Use GA to mimic batch size!
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps =30, # Chrysa added, 
        learning_rate = 2e-4, # Reduce to 2e-5 for long training runs
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=24):   0%|          | 0/6500 [00:00<?, ? examples/s]

In [15]:
def clear_gpu_memory():
    import torch, gc, os
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    print("GPU memory cleared.")

clear_gpu_memory()

GPU memory cleared.


### Section 3c: Show current memory stats

In [16]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA GB10. Max memory = 121.628 GB.
10.836 GB of memory reserved.


In [17]:
## ensure that the model will run in GPU
print(model.device)

cuda:0


In [18]:
def recursively_fix_device(module, device):
    if not hasattr(module, "device") or module.device is None:
        module.device = device
    for child in module.children():
        recursively_fix_device(child, device)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
recursively_fix_device(model, device)

bad = [n for n, m in model.named_modules() if getattr(m, "device", None) is None]
print("Modules without device:", bad)

Modules without device: []


### Section 3b continue: 

In [19]:
trainer_stats = trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 6,500 | Num Epochs = 1 | Total steps = 30
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 57,016,320 of 12,304,798,720 (0.46% trained)
wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: axelon-rep (axelon-rep-altae) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference, mcp, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


* Trackio project initialized: huggingface
* Trackio metrics will be synced to Hugging Face Dataset: axeledi/trackio-dataset
* Found existing space: https://huggingface.co/spaces/axeledi/trackio
* View dashboard by going to: https://axeledi-trackio.hf.space/


* Created new run: axeledi-1774478358


Step,Training Loss
1,1.656100
2,1.661100
3,1.660000
4,1.644900
5,1.520100
6,1.347000
7,1.126500
8,1.020200
9,0.835900
10,0.681500


* Run finished. Uploading logs to Trackio (please wait...)


KeyboardInterrupt: 

## Post-model

In [ ]:
# save to huggingface
model.push_to_hub(f"axeledi/{model_tag}-abnbTuned")

# Chrysa: For local saving as lora_model
model.save_pretrained(f"/home/akranis/tests/llm_bnb/cl_version_nonUV/tuned/lora_{model_tag}")  
tokenizer.save_pretrained(f"/home/akranis/tests/llm_bnb/cl_version_nonUV/tuned/lora_{model_tag}")

Saved model to https://huggingface.co/axeledi/Mistral-Nemo-Base-2407-bnb-4bit-abnbTuned


('/home/akranis/tests/llm_bnb/cl_version_nonUV/tuned/lora_Mistral-Nemo-Base-2407-bnb-4bit/tokenizer_config.json',
 '/home/akranis/tests/llm_bnb/cl_version_nonUV/tuned/lora_Mistral-Nemo-Base-2407-bnb-4bit/special_tokens_map.json',
 '/home/akranis/tests/llm_bnb/cl_version_nonUV/tuned/lora_Mistral-Nemo-Base-2407-bnb-4bit/tokenizer.json')

In [103]:
## save the TRN/VAL set for validation 
trn_dataset.to_json("/home/akranis/tests/llm_bnb/cl_version_nonUV/tuned/trn_set.json")
val_dataset.to_json("/home/akranis/tests/llm_bnb/cl_version_nonUV/tuned/val_set.json")

Creating json from Arrow format:   0%|          | 0/7 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/3 [00:00<?, ?ba/s]

13425478